# HW7 Mini Project — Your IFR Interpolator

Design your own interpolator. For compatible data, it should be continuous, pass through the
supplied IFR points, and be **monotone and convex within the node interval**. Explain your choice
of method and how you handle data that cannot satisfy all these conditions.

You may consult **Hagan and West's method** for inspiration.

### Implement your subclass

Write your class in `fixedincomelib/utilities/numerics.py`, inheriting from `Interpolator1D`.
Implement `interpolate` and `integrate`; leave the two optional gradient stubs as supplied.
The `MONOTONE_CONVEX` entry in `InterpMethod` and its factory mapping to
`Interpolator1DMonotoneConvex` are supplied. The one-node funding curve must be constant.

Each cell gives the product terms explicitly and calls the existing library APIs directly.
Within each pair, change only `'interpolation method'` from
`'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'` to `'MONOTONE_CONVEX'`; rebuilding the model selects
your class. Compare the PVs and briefly explain the differences. Use `FLAT` extrapolation
and the supplied IFR data; no calibration is needed.

Cell 1 includes the shared imports and data. Run cells **1, 3, 5** for the PCP results;
after completing your implementation, restart the kernel and run all six.



In [1]:
import pandas as pd
from fixedincomelib import (
    qfCreateData1D, qfCreateDataCollection,
    qfCreateBuildMethod, qfCreateModelBuildMethodCollection, qfCreateModel,
    qfCreateProductFRA, qfCreateProductRFRSwap, qfCreateProductOvernightIndexBasisSwap,
    qfCreateValuationParameters, qfCreateValuationParametersCollection,
    qfCreateValueReport,
)

# Supplied IFR point values, in annual decimals.
node_dates = ["2026-04-02", "2026-10-02", "2027-07-02",
              "2028-01-03", "2029-01-02", "2031-01-02"]
sofr_quotes = pd.DataFrame({"values": [0.025, 0.026, 0.028, 0.030, 0.035, 0.047]}, index=node_dates)
ff_quotes = pd.DataFrame({"values": [0.023, 0.0237, 0.0252, 0.0265, 0.0305, 0.0405]}, index=node_dates)
funding_quotes = pd.DataFrame({"values": [0.0005]}, index=["2027-01-04"])
data_collection = qfCreateDataCollection([
    qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-IFR", sofr_quotes),
    qfCreateData1D("INSTANTANEOUS FORWARD RATE", "FF-1B-IFR", ff_quotes),
    qfCreateData1D("INSTANTANEOUS FORWARD RATE", "SOFR-1B-FUNDING-IFR", funding_quotes),
])
curve_inputs = [
    ("YIELD_CURVE_INDEX", {"TARGET": "SOFR-1B", "INSTANTANEOUS FORWARD RATE": "SOFR-1B-IFR"}),
    ("YIELD_CURVE_INDEX", {"TARGET": "FF-1B", "INSTANTANEOUS FORWARD RATE": "FF-1B-IFR"}),
    ("YIELD_CURVE_FUNDING", {"TARGET": "SOFR-1B-FLAT", "REFERENCE INDEX": "SOFR-1B",
                            "INSTANTANEOUS FORWARD RATE": "SOFR-1B-FUNDING-IFR"}),
]


### Include an IFR curve plot and briefly explain how your method ensures monotonicity and convexity.

In [ ]:
# I build one model per method from the same IFR data and plot each index curve.
import numpy as np
import matplotlib.pyplot as plt

def build_curve_model(interpolation_method):
    methods = qfCreateModelBuildMethodCollection([
        qfCreateBuildMethod(kind, {**inputs,
                                   "INTERPOLATION METHOD": interpolation_method,
                                   "EXTRAPOLATION METHOD": "FLAT"})
        for kind, inputs in curve_inputs
    ])
    return qfCreateModel("2026-01-02", "YIELD_CURVE", data_collection, methods)

pcp_model = build_curve_model("PIECEWISE_CONSTANT_LEFT_CONTINUOUS")
mc_model = build_curve_model("MONOTONE_CONVEX")

ink, muted, grid = "#0b0b0b", "#52514e", "#e4e3df"
mc_color, pcp_color = "#2a78d6", "#eb6834"
times = np.linspace(0.0, 6.0, 1201)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, index in zip(axes, ["SOFR-1B", "FF-1B"]):
    pcp_curve = pcp_model.retrieve_model_component(index).state_data_interpolator
    mc_curve = mc_model.retrieve_model_component(index).state_data_interpolator
    node_t = mc_model.retrieve_model_component(index).state_data[0]
    node_f = mc_model.retrieve_model_component(index).state_data[1]
    ax.plot(times, [100 * pcp_curve.interpolate(x) for x in times],
            color=pcp_color, lw=2, drawstyle="default", label="Piecewise constant (left-continuous)")
    ax.plot(times, [100 * mc_curve.interpolate(x) for x in times],
            color=mc_color, lw=2, label="Monotone convex (mine)")
    ax.scatter(node_t, 100 * np.asarray(node_f), s=36, color=ink, zorder=3, label="Supplied IFR points")
    ax.set_title(f"{index} instantaneous forward curve", color=ink, fontsize=11, loc="left")
    ax.set_xlabel("Years from value date 2026-01-02", color=muted)
    ax.grid(True, color=grid, lw=0.8)
    ax.tick_params(colors=muted)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(grid)
axes[0].set_ylabel("IFR (%)", color=muted)
axes[0].legend(frameon=False, loc="lower right", labelcolor=ink)
plt.tight_layout()
plt.show()

# Numerical check of the shape on [t_0, t_N]: second differences >= 0 means convex,
# first differences >= 0 means monotone increasing.
for index in ["SOFR-1B", "FF-1B"]:
    curve = mc_model.retrieve_model_component(index).state_data_interpolator
    nodes = mc_model.retrieve_model_component(index).state_data
    grid_t = np.linspace(nodes[0][0], nodes[0][-1], 20001)
    values = np.array([curve.interpolate(x) for x in grid_t])
    print(f"{index}: passes through nodes = "
          f"{np.allclose([curve.interpolate(x) for x in nodes[0]], nodes[1], atol=1e-15)}, "
          f"monotone = {np.diff(values).min() >= -1e-15}, "
          f"convex = {np.diff(values, 2).min() >= -1e-12}")
print("One-node funding curve is constant:",
      len({round(mc_model.retrieve_model_component('SOFR-1B-FLAT').state_data_interpolator.interpolate(x), 15)
           for x in [0.0, 1.0, 3.0, 10.0]}) == 1)


### My method: a shape-preserving C1 quadratic spline (point-data version of Hagan–West)

**Why not Hagan–West as it is.** Their monotone convex method takes *discrete* forwards (an average rate over each interval) and builds a curve whose average over each interval matches them. Our inputs are *point* values of the instantaneous forward at each node, and the curve has to pass through them. So I keep their key idea, quadratic pieces with an extra breakpoint inside an interval when one quadratic cannot keep the right shape (their $\eta$ and region (iv)), and rebuild it for point data.

**Construction.** On $[t_0, t_N]$ the curve is made of quadratics, so its slope is piecewise linear.
1. *Slopes at the nodes.* With secant slopes $s_i = (f_i - f_{i-1})/(t_i - t_{i-1})$, the slope at an interior node is the harmonic mean $d_i = 2 s_i s_{i+1}/(s_i + s_{i+1})$ when both secants have the same sign, and $0$ otherwise. At the two end nodes I use $d_0 = 2s_1 - d_1$ and $d_N = 2s_N - d_{N-1}$.
2. *Shape on each interval.* With end slopes $d_{i-1}, d_i$ I place a breakpoint $\xi$ at fraction $\lambda$ of the interval. Requiring the curve to land exactly on $f_i$ fixes the slope at the breakpoint: $D = 2s_i - \lambda d_{i-1} - (1-\lambda) d_i$. When $d_{i-1} \le s_i \le d_i$, choosing $\lambda$ in $\left[\tfrac{\beta-\alpha}{\alpha+\beta}, \tfrac{2\beta}{\alpha+\beta}\right]$ (where $\alpha = s_i - d_{i-1}$, $\beta = d_i - s_i$) keeps $D$ between $d_{i-1}$ and $d_i$. The slope then only increases across the interval, so the piece is convex. I take the middle of that range.
3. *Integration* is exact: each piece is a quadratic, so the discount-factor integral $\int f$ is a closed-form cubic.

**Why it is monotone.** The harmonic mean of two same-signed numbers lies between them and is at most twice the smaller one, so every slope keeps the sign of the adjacent secants and stays between $0$ and $2s_i$. The slope at the breakpoint stays in that same range, so the slope never changes sign inside an interval. The curve never overshoots its two node values, so there are no spurious bumps, and no negative rates from positive data.

**Why it is convex.** For convex data the secants increase, so each interior $d_i$ lies strictly between $s_i$ and $s_{i+1}$, and every interval falls in the convex case of step 2. The slope is continuous at the nodes (the curve is $C^1$) and never decreases, so the curve is convex over the whole node range, not just piece by piece. The supplied SOFR and Fed Funds points have increasing secants, so both curves are monotone increasing and convex, as the check above confirms.

**Data that cannot satisfy everything.** I rank the requirements: continuity and passing through the nodes always hold; monotonicity on every interval always holds; convexity is given up only where the data itself is not convex. Where the secants turn over (a hump or an S-shape), I put the breakpoint at the midpoint and the curve stays monotone between the two nodes. If exactly collinear points sit next to curved ones, a $C^1$ convex curve does not exist, so I keep convexity and let the slope step up at that node instead.

**Outside the nodes** the curve is flat at $f_0$ and $f_N$ (FLAT extrapolation), matching the piecewise-constant interpolator, and a one-node curve such as the funding spread is a constant.


In [2]:
# 1. FRA — piecewise constant
fra_pcp_convention = {
    'effective date': '2026-05-05',
    'termination date': '2026-11-05',
    'projection index': 'SOFR-1B',
    'fixed rate': 0.032,
    'notional': 1000000,
    'pay or receive': 'receive',
    'accrual basis': 'ACT/360',
    'business day convention': 'F',
    'holiday convention': 'USGS',
    'value date': '2026-01-02',
    'funding index': 'SOFR-1B-FLAT',
    'interpolation method': 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS',
    'extrapolation method': 'FLAT',
}

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod(kind, {
        **inputs,
        "INTERPOLATION METHOD": fra_pcp_convention["interpolation method"],
        "EXTRAPOLATION METHOD": fra_pcp_convention["extrapolation method"],
    }) for kind, inputs in curve_inputs
])
fra_pcp_model = qfCreateModel(
    fra_pcp_convention["value date"], "YIELD_CURVE", data_collection, build_methods,
)
fra_pcp_product = qfCreateProductFRA(
    effective_date=fra_pcp_convention['effective date'],
    term_or_termination_date=fra_pcp_convention['termination date'],
    projection_index=fra_pcp_convention['projection index'],
    fixed_rate=fra_pcp_convention['fixed rate'],
    notional=fra_pcp_convention['notional'],
    pay_or_rec=fra_pcp_convention['pay or receive'],
    accrual_basis=fra_pcp_convention['accrual basis'],
    business_day_convention=fra_pcp_convention['business day convention'],
    holiday_convention=fra_pcp_convention['holiday convention'],
)
fra_pcp_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": fra_pcp_convention["funding index"],
    }),
])
fra_pcp_pv = qfCreateValueReport(fra_pcp_model, fra_pcp_product, fra_pcp_parameters, "PV")[0][1]
print("FRA | PCP PV:", fra_pcp_pv)


FRA | PCP PV: 2908.2866392379638


In [ ]:
# 2. FRA — your interpolator
fra_student_convention = {
    'effective date': '2026-05-05',
    'termination date': '2026-11-05',
    'projection index': 'SOFR-1B',
    'fixed rate': 0.032,
    'notional': 1000000,
    'pay or receive': 'receive',
    'accrual basis': 'ACT/360',
    'business day convention': 'F',
    'holiday convention': 'USGS',
    'value date': '2026-01-02',
    'funding index': 'SOFR-1B-FLAT',
    'interpolation method': 'MONOTONE_CONVEX',
    'extrapolation method': 'FLAT',
}

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod(kind, {
        **inputs,
        "INTERPOLATION METHOD": fra_student_convention["interpolation method"],
        "EXTRAPOLATION METHOD": fra_student_convention["extrapolation method"],
    }) for kind, inputs in curve_inputs
])
fra_student_model = qfCreateModel(
    fra_student_convention["value date"], "YIELD_CURVE", data_collection, build_methods,
)
fra_student_product = qfCreateProductFRA(
    effective_date=fra_student_convention['effective date'],
    term_or_termination_date=fra_student_convention['termination date'],
    projection_index=fra_student_convention['projection index'],
    fixed_rate=fra_student_convention['fixed rate'],
    notional=fra_student_convention['notional'],
    pay_or_rec=fra_student_convention['pay or receive'],
    accrual_basis=fra_student_convention['accrual basis'],
    business_day_convention=fra_student_convention['business day convention'],
    holiday_convention=fra_student_convention['holiday convention'],
)
fra_student_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": fra_student_convention["funding index"],
    }),
])
fra_student_pv = qfCreateValueReport(fra_student_model, fra_student_product, fra_student_parameters, "PV")[0][1]
print("FRA | Student PV:", fra_student_pv)
print("Student minus PCP:", fra_student_pv - fra_pcp_pv)


In [3]:
# 3. RFR swap — piecewise constant
swap_pcp_convention = {
    'effective date': '2026-04-02',
    'termination date': '2029-04-02',
    'overnight index': 'SOFR-1B',
    'fixed rate': 0.034,
    'notional': 1000000,
    'pay or receive': 'receive',
    'fixed accrual period': '1Y',
    'floating accrual period': '1Y',
    'accrual basis': 'ACT/360',
    'compounding method': 'compound',
    'floating spread': 0.0,
    'payment offset': '2D',
    'payment business day convention': 'F',
    'payment holiday convention': 'USGS',
    'value date': '2026-01-02',
    'funding index': 'SOFR-1B-FLAT',
    'interpolation method': 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS',
    'extrapolation method': 'FLAT',
}

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod(kind, {
        **inputs,
        "INTERPOLATION METHOD": swap_pcp_convention["interpolation method"],
        "EXTRAPOLATION METHOD": swap_pcp_convention["extrapolation method"],
    }) for kind, inputs in curve_inputs
])
swap_pcp_model = qfCreateModel(
    swap_pcp_convention["value date"], "YIELD_CURVE", data_collection, build_methods,
)
swap_pcp_product = qfCreateProductRFRSwap(
    effective_date=swap_pcp_convention['effective date'],
    term_or_termination_date=swap_pcp_convention['termination date'],
    on_index=swap_pcp_convention['overnight index'],
    fixed_rate=swap_pcp_convention['fixed rate'],
    notional=swap_pcp_convention['notional'],
    pay_or_rec=swap_pcp_convention['pay or receive'],
    accrual_period=swap_pcp_convention['fixed accrual period'],
    floating_leg_accrual_period=swap_pcp_convention['floating accrual period'],
    accrual_basis=swap_pcp_convention['accrual basis'],
    compounding_method=swap_pcp_convention['compounding method'],
    spread=swap_pcp_convention['floating spread'],
    payment_off_set=swap_pcp_convention['payment offset'],
    pay_business_day_convention=swap_pcp_convention['payment business day convention'],
    pay_holiday_convention=swap_pcp_convention['payment holiday convention'],
)
swap_pcp_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": swap_pcp_convention["funding index"],
    }),
])
swap_pcp_pv = qfCreateValueReport(swap_pcp_model, swap_pcp_product, swap_pcp_parameters, "PV")[0][1]
print("RFR swap | PCP PV:", swap_pcp_pv)


RFR swap | PCP PV: 6222.517513070154


In [ ]:
# 4. RFR swap — your interpolator
swap_student_convention = {
    'effective date': '2026-04-02',
    'termination date': '2029-04-02',
    'overnight index': 'SOFR-1B',
    'fixed rate': 0.034,
    'notional': 1000000,
    'pay or receive': 'receive',
    'fixed accrual period': '1Y',
    'floating accrual period': '1Y',
    'accrual basis': 'ACT/360',
    'compounding method': 'compound',
    'floating spread': 0.0,
    'payment offset': '2D',
    'payment business day convention': 'F',
    'payment holiday convention': 'USGS',
    'value date': '2026-01-02',
    'funding index': 'SOFR-1B-FLAT',
    'interpolation method': 'MONOTONE_CONVEX',
    'extrapolation method': 'FLAT',
}

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod(kind, {
        **inputs,
        "INTERPOLATION METHOD": swap_student_convention["interpolation method"],
        "EXTRAPOLATION METHOD": swap_student_convention["extrapolation method"],
    }) for kind, inputs in curve_inputs
])
swap_student_model = qfCreateModel(
    swap_student_convention["value date"], "YIELD_CURVE", data_collection, build_methods,
)
swap_student_product = qfCreateProductRFRSwap(
    effective_date=swap_student_convention['effective date'],
    term_or_termination_date=swap_student_convention['termination date'],
    on_index=swap_student_convention['overnight index'],
    fixed_rate=swap_student_convention['fixed rate'],
    notional=swap_student_convention['notional'],
    pay_or_rec=swap_student_convention['pay or receive'],
    accrual_period=swap_student_convention['fixed accrual period'],
    floating_leg_accrual_period=swap_student_convention['floating accrual period'],
    accrual_basis=swap_student_convention['accrual basis'],
    compounding_method=swap_student_convention['compounding method'],
    spread=swap_student_convention['floating spread'],
    payment_off_set=swap_student_convention['payment offset'],
    pay_business_day_convention=swap_student_convention['payment business day convention'],
    pay_holiday_convention=swap_student_convention['payment holiday convention'],
)
swap_student_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": swap_student_convention["funding index"],
    }),
])
swap_student_pv = qfCreateValueReport(swap_student_model, swap_student_product, swap_student_parameters, "PV")[0][1]
print("RFR swap | Student PV:", swap_student_pv)
print("Student minus PCP:", swap_student_pv - swap_pcp_pv)


In [4]:
# 5. Overnight basis swap — piecewise constant
basis_pcp_convention = {
    'effective date': '2026-07-02',
    'termination date': '2028-07-03',
    'overnight index leg 1': 'SOFR-1B',
    'overnight index leg 2': 'FF-1B',
    'spread over leg 1': 0.001,
    'notional': 1000000,
    'pay or receive leg 1': 'receive',
    'accrual period leg 1': '3M',
    'accrual period leg 2': '3M',
    'accrual basis': 'ACT/360',
    'compounding method': 'compound',
    'payment offset': '2D',
    'payment business day convention': 'F',
    'payment holiday convention': 'USGS',
    'value date': '2026-01-02',
    'funding index': 'SOFR-1B-FLAT',
    'interpolation method': 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS',
    'extrapolation method': 'FLAT',
}

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod(kind, {
        **inputs,
        "INTERPOLATION METHOD": basis_pcp_convention["interpolation method"],
        "EXTRAPOLATION METHOD": basis_pcp_convention["extrapolation method"],
    }) for kind, inputs in curve_inputs
])
basis_pcp_model = qfCreateModel(
    basis_pcp_convention["value date"], "YIELD_CURVE", data_collection, build_methods,
)
basis_pcp_product = qfCreateProductOvernightIndexBasisSwap(
    effective_date=basis_pcp_convention['effective date'],
    term_or_termination_date=basis_pcp_convention['termination date'],
    on_index_leg_1=basis_pcp_convention['overnight index leg 1'],
    on_index_leg_2=basis_pcp_convention['overnight index leg 2'],
    spread_over_leg_1=basis_pcp_convention['spread over leg 1'],
    notional=basis_pcp_convention['notional'],
    pay_or_rec_leg_1=basis_pcp_convention['pay or receive leg 1'],
    accrual_period_1=basis_pcp_convention['accrual period leg 1'],
    accrual_period_2=basis_pcp_convention['accrual period leg 2'],
    accrual_basis=basis_pcp_convention['accrual basis'],
    compounding_method=basis_pcp_convention['compounding method'],
    payment_off_set=basis_pcp_convention['payment offset'],
    pay_business_day_convention=basis_pcp_convention['payment business day convention'],
    pay_holiday_convention=basis_pcp_convention['payment holiday convention'],
)
basis_pcp_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": basis_pcp_convention["funding index"],
    }),
])
basis_pcp_pv = qfCreateValueReport(basis_pcp_model, basis_pcp_product, basis_pcp_parameters, "PV")[0][1]
print("Overnight basis swap | PCP PV:", basis_pcp_pv)


Overnight basis swap | PCP PV: 8351.348908181993


In [ ]:
# 6. Overnight basis swap — your interpolator
basis_student_convention = {
    'effective date': '2026-07-02',
    'termination date': '2028-07-03',
    'overnight index leg 1': 'SOFR-1B',
    'overnight index leg 2': 'FF-1B',
    'spread over leg 1': 0.001,
    'notional': 1000000,
    'pay or receive leg 1': 'receive',
    'accrual period leg 1': '3M',
    'accrual period leg 2': '3M',
    'accrual basis': 'ACT/360',
    'compounding method': 'compound',
    'payment offset': '2D',
    'payment business day convention': 'F',
    'payment holiday convention': 'USGS',
    'value date': '2026-01-02',
    'funding index': 'SOFR-1B-FLAT',
    'interpolation method': 'MONOTONE_CONVEX',
    'extrapolation method': 'FLAT',
}

build_methods = qfCreateModelBuildMethodCollection([
    qfCreateBuildMethod(kind, {
        **inputs,
        "INTERPOLATION METHOD": basis_student_convention["interpolation method"],
        "EXTRAPOLATION METHOD": basis_student_convention["extrapolation method"],
    }) for kind, inputs in curve_inputs
])
basis_student_model = qfCreateModel(
    basis_student_convention["value date"], "YIELD_CURVE", data_collection, build_methods,
)
basis_student_product = qfCreateProductOvernightIndexBasisSwap(
    effective_date=basis_student_convention['effective date'],
    term_or_termination_date=basis_student_convention['termination date'],
    on_index_leg_1=basis_student_convention['overnight index leg 1'],
    on_index_leg_2=basis_student_convention['overnight index leg 2'],
    spread_over_leg_1=basis_student_convention['spread over leg 1'],
    notional=basis_student_convention['notional'],
    pay_or_rec_leg_1=basis_student_convention['pay or receive leg 1'],
    accrual_period_1=basis_student_convention['accrual period leg 1'],
    accrual_period_2=basis_student_convention['accrual period leg 2'],
    accrual_basis=basis_student_convention['accrual basis'],
    compounding_method=basis_student_convention['compounding method'],
    payment_off_set=basis_student_convention['payment offset'],
    pay_business_day_convention=basis_student_convention['payment business day convention'],
    pay_holiday_convention=basis_student_convention['payment holiday convention'],
)
basis_student_parameters = qfCreateValuationParametersCollection([
    qfCreateValuationParameters("FUNDING INDEX PARAMETER", {
        "FUNDING INDEX": basis_student_convention["funding index"],
    }),
])
basis_student_pv = qfCreateValueReport(basis_student_model, basis_student_product, basis_student_parameters, "PV")[0][1]
print("Overnight basis swap | Student PV:", basis_student_pv)
print("Student minus PCP:", basis_student_pv - basis_pcp_pv)


In [ ]:
# Side-by-side PVs for the three products.
summary = pd.DataFrame({
    "Product": ["FRA", "RFR swap", "Overnight basis swap"],
    "PCP PV": [fra_pcp_pv, swap_pcp_pv, basis_pcp_pv],
    "Monotone convex PV": [fra_student_pv, swap_student_pv, basis_student_pv],
})
summary["Difference"] = summary["Monotone convex PV"] - summary["PCP PV"]
summary.round(2)


### Why the PVs differ

**The common cause.** The left-continuous piecewise-constant curve gives each whole interval $(t_{i-1}, t_i]$ the value at its *right* node. These curves slope upward, so that is the highest value on the interval, and every forward between nodes comes out biased up. My curve rises gradually from $f_{i-1}$ to $f_i$, so it agrees at the nodes and sits *below* the step curve between them. The funding spread is a single constant node under both methods, so all three differences come from the SOFR and Fed Funds curves. Note that SOFR-1B-FLAT references SOFR, so a lower SOFR curve also means higher discount factors.

- **FRA (receive 3.2% fixed on 6M SOFR, May to Nov 2026), +354.** Most of the period sits between the 0.25Y and 0.75Y nodes, where the step curve uses the 2.6% right-node value throughout, and its last month already takes the 2.8% of the next node. The projected term rate falls from 2.618% to 2.548%, about 7bp. Receiving fixed gains roughly $N \times \tau \times 7\text{bp} \times D \approx 1{,}000{,}000 \times 0.511 \times 0.0007 \times 0.978 \approx 350$.
- **3Y RFR swap (receive 3.4% fixed), +6,628.** This is the largest gap. The projected annual rates fall from 2.70%, 3.08% and 3.81% to 2.60%, 2.90% and 3.37%. The third year spans the 3Y node, and the step curve jumps straight to the 4.7% five-year value just after it, while my curve climbs smoothly. The floating leg's PV falls by about 6,290, and the fixed leg gains about 340 from the higher discount factors.
- **2Y basis swap (receive SOFR + 10bp, pay Fed Funds), −724.** The value comes from the SOFR minus Fed Funds gap. Under the step curve that gap jumps between node values (22.7, 27.8, 34.8 and 44.6bp), always taking the wider spread at the right node. My curves move smoothly, so the gap widens gradually from 21bp to 39bp and is smaller on average. A smaller gap is worth less to the receiver of SOFR. The 10bp spread leg barely changes (+3), since it depends only on discounting.

**Takeaway.** Piecewise-constant forwards are a crude approximation that systematically overstates forwards on a rising curve, and the overstatement grows with the steepness and the gap between nodes. A smooth, shape-preserving curve removes that bias while still matching every input point.
